In [1]:
import operator
import asyncio

from typing import Annotated, List, TypedDict

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send 

from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate


In [2]:
fanout_worker_llm = ChatGroq(model="openai/gpt-oss-20b")
editor_base_llm = ChatGroq(model="openai/gpt-oss-120b")

In [3]:
class OverallState(TypedDict):
    topic: str 
    subtopics: List[str] 
    research_results: Annotated[List[str], operator.add]
    final_report: str 

class ResearchState(TypedDict): # private state
    subtopic: str 

In [4]:
class SubtopicPlan(BaseModel):
    subtopics: List[str] = Field(..., description="List of subtopics to research")

In [5]:
async def generate_subtopics(state: OverallState):
    topic = state["topic"]

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Break the given research topic into clear, specific subtopics for a comprehensive report."),
        ("human", "Topic: {topic}"),
    ])

    planner_chain = prompt | fanout_worker_llm.with_structured_output(SubtopicPlan)

    plan: SubtopicPlan = await planner_chain.ainvoke({
        "topic": topic,
    })

    return {
        "subtopics": plan.subtopics
    }

In [6]:
async def research_subtopic(state: ResearchState):
    subtopic = state["subtopic"]

    print(subtopic)

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Provide a concise, insightful 1-paragraph research summary of the following subtopic with key facts and insights."),
        ("human", "Subtopic: {subtopic}"),
    ])

    researcher_chain = prompt | fanout_worker_llm 

    result = await researcher_chain.ainvoke({
        "subtopic": subtopic,
    })

    print("Subtopic Research Result: ", result.content)

    finding = f"### {subtopic} ###\n{result.content}"

    return {
        "research_results": [finding]
    }

In [7]:
async def compile_report(state: OverallState):
    combined_notes = "\n\n".join(state["research_results"])

    print(combined_notes)

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a senior research editor. Synthesize the following subtopic research notes "
            "into a well-structured, cohesive executive report with an introduction and conclusion.",
        ),
        ("human", "Main Topic: {topic}\nSubtopic Research Notes: {combined_notes}"),
    ])

    editor_chain = prompt | editor_base_llm 

    response = await editor_chain.ainvoke({
        "topic": state["topic"],
        "combined_notes": combined_notes,
    })

    return {
        "final_report": response.content
    }

In [8]:
def map_to_research(state: OverallState):
    return [
        Send(
            "research_subtopic",
            {"subtopic": s} # private state
        )
        for s in state["subtopics"]
    ]


In [9]:
builder = StateGraph(OverallState)

builder.add_node("generate_subtopics", generate_subtopics)
builder.add_node("research_subtopic", research_subtopic)
builder.add_node("compile_report", compile_report)

builder.add_edge(START, "generate_subtopics") 
builder.add_conditional_edges("generate_subtopics", map_to_research)
builder.add_edge("research_subtopic", "compile_report") 
builder.add_edge("compile_report", END)

graph = builder.compile()

In [10]:
initial_state = {
    "topic": "The evolution of GPU in the last 10 years",
    "research_results": [],
    "final_report": "",
    "subtopics": [],
}

In [11]:
result = await graph.ainvoke(initial_state)

Historical milestones and key GPU architectures from 2014-2024
Performance evolution: FLOPS, memory bandwidth, core counts, and clock speeds
Architectural innovations: ray tracing cores, tensor cores, AI acceleration, and energy efficiency
Market dynamics: major players (NVIDIA, AMD, Intel, ARM), product lines, and pricing strategies
Software ecosystem: drivers, APIs (CUDA, DirectX, Vulkan), and compiler optimizations
Gaming impact: graphics quality, real-time ray tracing, and VR/AR advancements
AI and deep learning: GPU usage trends, training vs inference workloads, and specialized accelerators
Power consumption and thermal design: TDP changes, cooling solutions, and sustainability considerations
Emerging technologies: Heterogeneous integration, chiplet design, and 3D stacking
Future outlook: predicted architectural directions, competition, and potential disruptions
Subtopic Research Result:  Emerging semiconductor packaging is rapidly shifting from monolithic die to modular, vertical

In [12]:
print(result["final_report"])

**Executive Report – The Evolution of GPUs (2014‑2024)**  
*Prepared for senior leadership and technology‑strategy teams*  

---

## 1. Introduction  

Graphics Processing Units have undergone a profound transformation in the past decade. Once specialized rasterizers, today’s GPUs are heterogeneous compute engines that simultaneously drive high‑end gaming, real‑time ray tracing, AI inference/training, and data‑center workloads. This report condenses the most salient technical milestones, performance trends, market forces, and emerging technologies that have reshaped the GPU landscape from 2014 to 2024, and projects the architectural directions likely to dominate the next five‑year horizon.

---

## 2. Historical Milestones & Key Architectures (2014‑2024)

| Year | Vendor | Architecture | Notable Innovations | Impact |
|------|--------|--------------|---------------------|--------|
| **2015** | NVIDIA | **Pascal (GP104)** | 16 GB HBM2, 5 TFLOPS FP32, 14 nm FinFET | Set new price‑perform

In [13]:
print(result["subtopics"])

['Historical milestones and key GPU architectures from 2014-2024', 'Performance evolution: FLOPS, memory bandwidth, core counts, and clock speeds', 'Architectural innovations: ray tracing cores, tensor cores, AI acceleration, and energy efficiency', 'Market dynamics: major players (NVIDIA, AMD, Intel, ARM), product lines, and pricing strategies', 'Software ecosystem: drivers, APIs (CUDA, DirectX, Vulkan), and compiler optimizations', 'Gaming impact: graphics quality, real-time ray tracing, and VR/AR advancements', 'AI and deep learning: GPU usage trends, training vs inference workloads, and specialized accelerators', 'Power consumption and thermal design: TDP changes, cooling solutions, and sustainability considerations', 'Emerging technologies: Heterogeneous integration, chiplet design, and 3D stacking', 'Future outlook: predicted architectural directions, competition, and potential disruptions']


In [14]:
print(result["research_results"])

['### Historical milestones and key GPU architectures from 2014-2024 ###\nFrom 2014‑2024 the GPU landscape evolved from memory‑bandwidth‑centric designs to AI‑first, ray‑tracing‑ready accelerators that blur the line between graphics and compute. Nvidia’s **Pascal (GP104/GPU) in 2015** pushed 16\u202fGB HBM2 memory and 5\u202fTFLOPS FP32, setting a new price‑performance bar; it was followed by **Volta (2017)**, introducing 8\u202fTensor Cores and 4.5\u202fTFLOPS FP64, a leap for deep‑learning workloads. The **Turing (2018)** family added RT cores for hardware‑accelerated ray tracing and further Tensor Cores, while **Ampere (2020)** doubled FP32 performance and added third‑generation Tensor Cores, powering the RTX 30‑series. Nvidia’s **Ada Lovelace (2022)** introduced 4th‑gen Tensor Cores, a new RT core architecture, and the first GPUs with a 3‑D‑VNNI instruction set, while **Hopper (2024)** promises 10‑fold AI throughput and a new Hopper GPU‑on‑Chip (GOC) for ultra‑low latency inference